In [ ]:
# Importing Pandas
import pandas as pd

In [ ]:
# Importing Transactions and Customers record
transactions = pd.read_csv("data/transactions.csv")
customers = pd.read_csv("data/customers.csv")

print(transactions)

In [ ]:
# Cleaning Transaction Dataset 

# Convert Amount to Numeric Data Type 
transactions["amount"] = (
    pd.to_numeric(transactions["amount"], errors="coerce")
)

# Filling missing amount with median 
median_amount = (
    transactions["amount"].median()
)

transactions["amount"] = (
    transactions["amount"].fillna(median_amount)
)

# Removing Duplicate Transaction ID 
transactions = (
    transactions.drop_duplicates(subset=["transaction_id"])
)

# Convert Dates 
transactions["date"] = (
    pd.to_datetime(transactions["date"])
)

In [19]:
# Merging Customers with Transactions DataSet 
df = transactions.merge(
    customers,
    on="customer_id",
    how="left"
)

# Rearranging Columns 
com_df = df[
    ["transaction_id", "customer_id", "age", "location", "transaction_type", "amount", "status", "failed_attempts", "account_age_days", "date"]
]

In [ ]:
# Feature Engineering 

# Checking high value 
com_df["is_high_value"] = (
    com_df["amount"] > 100000
).astype(int)

# Is New Account 
com_df["is_new_account"] = (
    com_df["account_age_days"] < 90
).astype(int)

# Failed Attempts 
com_df["multiple_failures"] = (
    com_df["failed_attempts"] >= 3
).astype(int)

# Weekday 
com_df["weekday"] = (
    com_df["date"].dt.day_name()
)

print(com_df)

In [ ]:
# Creating a Basic Risk Score 

com_df["risk_score"] = (
    com_df["is_high_value"] + com_df["is_new_account"] + com_df["multiple_failures"]
)

def classifyRisk(score:int) -> str:
    if score >= 2:
        return "High"
    if score == 1:
        return "Medium"
    return "Low"

com_df["risk_level"] = (
    com_df["risk_score"].apply(classifyRisk)
)

print(com_df)

In [23]:
# Export DataSet 

com_df.to_csv("data/complete_dataframe.csv", index=False)